
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/molecular_docking_tutorial/blob/main/practical_docking/01_Structure_Search.ipynb)

# 타겟 구조 탐색 & 최적 PDB 선택

## 목적

분자 도킹(Molecular Docking)을 수행하기 전, **가장 적합한 PDB 구조를 선택**하는 것이 매우 중요합니다.
같은 타겟 단백질이라도 PDB 구조에 따라 도킹 결과가 크게 달라질 수 있기 때문입니다.

## 이론적 배경

### 왜 구조 선택이 중요한가?
- **Resolution**: 해상도가 높을수록(숫자가 낮을수록) 원자 좌표가 정확
- **Co-crystal ligand**: 리간드가 결합된 구조는 결합 부위가 "열린" 형태
- **Conformation**: Active/Inactive 상태에 따라 결합 부위 형태가 다름

### Re-docking 검증
선택한 구조가 도킹에 적합한지 검증하는 표준 방법:
1. co-crystal 리간드의 SMILES에서 새로 3D 좌표 생성
2. 동일 구조에 re-docking
3. Crystal pose와 docked pose의 RMSD 비교
4. **RMSD < 2.0Å이면 도킹 프로토콜 신뢰 가능**

## 워크플로우
1. PDB 구조 검색 & 비교
2. 구조 정렬 (Align)
3. Docking Box 설정 & 시각화
4. 전체 구조 re-docking
5. 분석 (Score, RMSD, ProLIF 상호작용)
6. 최적 구조 판정


## 0. 환경 설정


### 환경 설치

Google Colab이나 Docker에서 필요한 Python 패키지와 도킹 바이너리(smina)를 설치합니다.
이미 설치되어 있으면 건너뛰어도 됩니다.


In [ ]:
#@title 환경 설치 (처음 1회) {display-mode: "form"}
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

# Core packages
pip_install('rdkit', 'gemmi', 'openbabel-wheel')
pip_install('pdbfixer', 'openmm')
pip_install('py3Dmol', 'prolif', 'MDAnalysis')
pip_install('seaborn', 'pandas', 'matplotlib', 'requests')

# pymol (optional — Colab/x86_64 only)
try:
    pip_install('pymol-open-source')
except: pass

# smina binary
BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    import stat, urllib.request
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print(f'smina: {smina_path}')
print('Done.')


### 라이브러리 로드

분자 도킹 파이프라인에 필요한 라이브러리를 불러옵니다:
- **RDKit**: 분자 조작, 핑거프린트, SMILES 처리
- **OpenBabel/Meeko**: PDBQT 포맷 변환
- **PDBFixer/OpenMM**: 단백질 구조 수정 (누락 원자, 수소 추가)
- **MDAnalysis**: 구조 정렬, 좌표 계산
- **ProLIF**: 단백질-리간드 상호작용 분석
- **py3Dmol**: 3D 구조 시각화


In [ ]:
#@title 라이브러리 로드 {display-mode: "form"}
import warnings; warnings.filterwarnings('ignore')
import os, subprocess, urllib.request, time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import py3Dmol
import requests
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolAlign
from rdkit.Chem import SanitizeFlags
from openbabel import pybel
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda
from MDAnalysis.analysis import align as mda_align
import prolif as plf

# BIN_DIR은 이전 셀에서 설정됨
WORK_DIR = '/content/structure_search' if os.path.exists('/content') else os.path.join(os.path.expanduser('~'), 'structure_search')
os.makedirs(WORK_DIR, exist_ok=True)

print('All libraries loaded.')


## 1. PDB 구조 검색

타겟 이름을 입력하면 RCSB PDB에서 X-ray 구조를 검색합니다.


### PDB 검색

RCSB PDB Search API를 사용하여 타겟 단백질의 X-ray 결정 구조를 검색합니다.
리간드가 결합된 구조만 필터링하고, resolution이 좋은 순서로 정렬합니다.


In [ ]:
#@title 1-1. 타겟 검색 {display-mode: "form"}
TARGET_NAME = "TYK2"  #@param {type:"string"}
MAX_RESULTS = 5      #@param {type:"integer"}

search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
# UniProt 기반 정확한 타겟 검색
# 잘 알려진 타겟은 UniProt ID로 검색하면 정확합니다
UNIPROT_MAP = {
    "TYK2": {"uniprot": "P29597", "keyword": "JH2 pseudokinase"},
    "EGFR": {"uniprot": "P00533", "keyword": ""},
    "ABL1": {"uniprot": "P00519", "keyword": ""},
    "JAK2": {"uniprot": "O60674", "keyword": ""},
}

target_info = UNIPROT_MAP.get(TARGET_NAME.upper(), None)

if target_info:
    # UniProt ID 기반 정확한 검색
    nodes = [
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                        "operator": "exact_match", "value": target_info["uniprot"]}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "exptl.method", "operator": "exact_match",
                        "value": "X-RAY DIFFRACTION"}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_entry_info.nonpolymer_entity_count",
                        "operator": "greater", "value": 0}}
    ]
    # JH2 등 추가 키워드가 있으면 full_text 필터 추가
    if target_info["keyword"]:
        nodes.append({"type": "terminal", "service": "full_text",
                      "parameters": {"value": target_info["keyword"]}})
else:
    # 일반 검색 (UniProt 매핑이 없는 경우)
    nodes = [
        {"type": "terminal", "service": "full_text",
         "parameters": {"value": f"{TARGET_NAME} kinase"}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "exptl.method", "operator": "exact_match",
                        "value": "X-RAY DIFFRACTION"}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_entry_info.nonpolymer_entity_count",
                        "operator": "greater", "value": 0}}
    ]

query = {
    "query": {
        "type": "group", "logical_operator": "and",
        "nodes": nodes
    },
    "return_type": "entry",
    "request_options": {
        "results_content_type": ["experimental"],
        "sort": [{"sort_by": "rcsb_entry_info.resolution_combined", "direction": "asc"}],
        "paginate": {"start": 0, "rows": MAX_RESULTS}
    }
}

try:
    resp = requests.post(search_url, json=query, timeout=30)
    resp.raise_for_status()
    pdb_ids = [hit["identifier"] for hit in resp.json().get("result_set", [])]
except Exception as e:
    print(f"PDB search failed: {e}")
    pdb_ids = []
if not pdb_ids:
    print("WARNING: No structures found. Check TARGET_NAME.")
pdb_ids = pdb_ids[:MAX_RESULTS]  # limit early to avoid slow API calls
print(f'{TARGET_NAME}: {len(pdb_ids)} entries (limited to {MAX_RESULTS})')


### 구조 메타데이터 수집

각 PDB 엔트리의 상세 정보를 가져옵니다:
- **Resolution**: 구조의 해상도 (낮을수록 정밀, 2.5Å 이하 권장)
- **Ligand**: co-crystal 리간드 (도킹 검증에 사용)
- **Year**: 등록 연도 (최신 구조가 보통 더 정밀)


In [ ]:
#@title 1-2. 구조 메타데이터 수집 & 정렬 {display-mode: "form"}
struct_data = []
print(f'Fetching metadata for top entries...', flush=True)

for pdb_id in pdb_ids[:MAX_RESULTS]:
    try:
        info = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}", timeout=5).json()
        res = info.get("rcsb_entry_info", {}).get("resolution_combined", [None])[0]
        year = info.get("rcsb_accession_info", {}).get("deposit_date", "")[:4]
        title = info.get("struct", {}).get("title", "")

        # Get ligand info (iterate all nonpolymer entities)
        lig_id = ""
        exclude_ligs = {'HOH','WAT','SO4','GOL','PEG','EDO','CL','NA','MG','ZN','CA','ACT','BME','DMS','PG4'}
        entity_ids = info.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", ["1"])
        for eid in entity_ids:
            try:
                lig_resp = requests.get(f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{eid}", timeout=5)
                if lig_resp.status_code == 200:
                    comp_id = lig_resp.json().get("pdbx_entity_nonpoly", {}).get("comp_id", "")
                    if comp_id and comp_id not in exclude_ligs:
                        lig_id = comp_id
                        break
            except: pass

        if res is not None and lig_id:
            struct_data.append({
                "PDB": pdb_id, "Resolution": round(res, 2),
                "Ligand": lig_id, "Year": int(year) if year else 0,
                "Title": title[:60]
            })
    except: pass

if not struct_data:
    print('WARNING: No structures with ligands found. Try a different target.')
    structs_df = pd.DataFrame(columns=['PDB','Resolution','Ligand','Year','Title'])
else:
    structs_df = pd.DataFrame(struct_data).sort_values("Resolution").head(MAX_RESULTS).reset_index(drop=True)
structs_df.index += 1
print(f'\n{len(structs_df)} structures with ligands (sorted by resolution):')
structs_df


### Resolution 비교 차트

검색된 구조들의 해상도를 바 차트로 비교합니다.
- **초록**: Resolution < 2.0Å (매우 좋음)
- **노랑**: 2.0~2.5Å (좋음)
- **빨강**: > 2.5Å (도킹에 부적합할 수 있음)


In [ ]:
#@title 1-3. Resolution 비교 차트 {display-mode: "form"}
if len(structs_df) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#E53935' if r > 2.5 else '#43A047' if r < 2.0 else '#FFB300' for r in structs_df['Resolution']]
    ax.barh(structs_df['PDB'], structs_df['Resolution'], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Resolution (Å)')
    ax.set_title(f'{TARGET_NAME} PDB Structures by Resolution')
    ax.axvline(x=2.5, color='red', linestyle='--', alpha=0.5, label='2.5 Å threshold')
    ax.invert_yaxis()
    ax.legend()
    for i, (pdb, res) in enumerate(zip(structs_df['PDB'], structs_df['Resolution'])):
        ax.text(res + 0.05, i, f'{res:.1f}Å', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()


## 2. 구조 선택 & 준비

검색 결과에서 비교할 구조를 선택합니다. 기본값은 상위 5개입니다.


### 비교할 구조 선택

Resolution이 좋은 상위 N개를 자동 선택하거나, 직접 PDB 코드를 입력합니다.
예: `6NZP, 6NZQ, 5T01` (쉼표로 구분)


In [ ]:
#@title 2-1. 비교할 PDB 선택 {display-mode: "form"}
# 상위 N개 자동 선택, 또는 직접 입력
AUTO_TOP_N = 5  #@param {type:"integer"}
MANUAL_PDB_LIST = ""  #@param {type:"string"}

if MANUAL_PDB_LIST.strip():
    SELECTED_PDBS = [p.strip().upper() for p in MANUAL_PDB_LIST.split(',')]
else:
    SELECTED_PDBS = structs_df['PDB'].head(AUTO_TOP_N).tolist()

print(f'Selected {len(SELECTED_PDBS)} structures: {SELECTED_PDBS}')


### 구조 준비 (Fetch → Clean → Fix)

각 PDB 구조에 대해:
1. **Fetch**: RCSB에서 PDB 파일 다운로드
2. **Clean**: 지정 chain만 추출, 물 분자/이온 제거
3. **Fix**: PDBFixer로 누락 잔기/원자 복구, pH 7.4에서 수소 추가
4. **PDBQT**: 도킹 엔진(smina)이 읽을 수 있는 포맷으로 변환

> **PDBQT 변환**: Receptor는 OpenBabel로 생성 후 ROOT/BRANCH 태그를 제거 (rigid receptor).
> Ligand는 SMILES → RDKit 3D → Meeko로 PDBQT 생성.


In [ ]:
#@title 2-2. 구조 준비 (fetch → clean → fix) {display-mode: "form"}
os.chdir(WORK_DIR)

structures = {}  # pdb_id -> {prot_H, lig_mol2, lig_H, rec_qt, lig_qt, ...}

for i, pdb_id in enumerate(SELECTED_PDBS):
    pdb_id = pdb_id.lower()
    print(f'[{i+1}/{len(SELECTED_PDBS)}] {pdb_id.upper()}...', end=' ', flush=True)

    pdb_dir = os.path.join(WORK_DIR, pdb_id)
    os.makedirs(pdb_dir, exist_ok=True)

    try:
        # Fetch PDB
        pdb_path = os.path.join(pdb_dir, f'{pdb_id}.pdb')
        if not os.path.exists(pdb_path):
            urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)

        # Extract protein chain A + ligand using MDAnalysis
        u = mda.Universe(pdb_path)
        prot_sel = u.select_atoms('protein and chainID A')
        lig_sel = u.select_atoms('not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4 and chainID A')
        if len(lig_sel) < 3:
            lig_sel = u.select_atoms('not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4')

        clean_pdb = os.path.join(pdb_dir, f'{pdb_id}_clean.pdb')
        prot_sel.write(clean_pdb)

        # Ligand → MOL2 via openbabel
        lig_pdb_tmp = os.path.join(pdb_dir, f'{pdb_id}_lig_tmp.pdb')
        lig_mol2 = os.path.join(pdb_dir, f'{pdb_id}_lig.mol2')
        lig_sel.write(lig_pdb_tmp)
        mol = list(pybel.readfile(format='pdb', filename=lig_pdb_tmp))[0]
        out = pybel.Outputfile(filename=lig_mol2, format='mol2', overwrite=True)
        out.write(mol); out.close()
        os.remove(lig_pdb_tmp)

        # PDBFixer
        prot_H = os.path.join(pdb_dir, f'{pdb_id}_clean_H.pdb')
        fixer = PDBFixer(filename=clean_pdb)
        fixer.findMissingResidues(); fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues(); fixer.removeHeterogens(True)
        fixer.findMissingAtoms(); fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.4)
        with open(prot_H, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)

        # Ligand + H
        lig_H = os.path.join(pdb_dir, f'{pdb_id}_lig_H.mol2')
        lmol = list(pybel.readfile(format='mol2', filename=lig_mol2))[0]
        lmol.addh()
        lout = pybel.Outputfile(filename=lig_H, format='mol2', overwrite=True)
        lout.write(lmol); lout.close()

        structures[pdb_id] = {
            'pdb_path': pdb_path,
            'clean_pdb': clean_pdb,
            'prot_H': prot_H,
            'lig_mol2': lig_mol2,
            'lig_H': lig_H,
            'dir': pdb_dir,
        }
        print('OK')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\n=== {len(structures)}/{len(SELECTED_PDBS)} structures prepared ===')


## 3. 구조 정렬 (Align)

첫 번째 구조를 기준으로 모든 구조를 정렬하고, RMSD 매트릭스를 계산합니다.


### 구조 정렬 & RMSD 매트릭스

첫 번째 구조를 기준(reference)으로 모든 구조를 Cα 원자 기준으로 정렬합니다.
- **RMSD**: 정렬 후 Cα 원자 간 평균 편차 (Å)
- RMSD가 낮을수록 구조적으로 유사
- 키나아제의 ATP binding site는 보존되어 있어 정렬이 가능합니다


In [ ]:
#@title 3-1. 구조 정렬 & RMSD 매트릭스 {display-mode: "form"}
from pymol import cmd as pcmd

pdb_ids_list = list(structures.keys())
names = pdb_ids_list
ref_name = names[0]

pcmd.delete('all')

# Load clean protein (chain A only) + original PDB for ligand
for name in names:
    s = structures[name]
    # Load cleaned protein (already chain A only)
    pcmd.load(s['prot_H'], f'{name}_prot')
    # Load original PDB for ligand extraction
    pcmd.load(s['pdb_path'], f'{name}_full')
    pcmd.remove(f'{name}_full and not (organic and chain A)')
    pcmd.remove(f'{name}_full and resn HOH+WAT+SOL+GOL+PEG+EDO+PG4+DMS+ACT+BME+CL+NA+MG+ZN+CA+SO4')

# CE-Align proteins
align_rmsds = {}
for name in names:
    if name == ref_name:
        align_rmsds[(ref_name, ref_name)] = 0.0
        continue
    try:
        result = pcmd.cealign(f'{ref_name}_prot', f'{name}_prot')
        rmsd_val = float(result['RMSD']) if 'RMSD' in result else float(list(result.values())[0])
        align_rmsds[(ref_name, name)] = round(rmsd_val, 2)
        align_rmsds[(name, ref_name)] = round(rmsd_val, 2)
        
        # Apply same transformation matrix to ligand
        pcmd.matrix_copy(f'{name}_prot', f'{name}_full')
        
        print(f'{name} aligned to {ref_name}: RMSD={rmsd_val:.2f} A')
    except Exception as e:
        print(f'{name} align failed: {e}')

# Save aligned protein + ligand
for name in names:
    s = structures[name]
    aligned_prot = s['prot_H'].replace('.pdb', '_aligned.pdb')
    aligned_lig = s['lig_mol2'].replace('.mol2', '_aligned.mol2')
    
    pcmd.save(aligned_prot, f'{name}_prot')
    pcmd.save(aligned_lig, f'{name}_full')
    
    structures[name]['prot_H_aligned'] = aligned_prot
    structures[name]['lig_aligned'] = aligned_lig

# Full pairwise RMSD
for i, id1 in enumerate(names):
    for j, id2 in enumerate(names):
        if i == j:
            align_rmsds[(id1, id2)] = 0.0
        elif (id1, id2) not in align_rmsds:
            try:
                result = pcmd.cealign(f'{id1}_prot', f'{id2}_prot')
                rv = float(result['RMSD']) if 'RMSD' in result else float(list(result.values())[0])
                align_rmsds[(id1, id2)] = round(rv, 2)
                align_rmsds[(id2, id1)] = round(rv, 2)
            except:
                align_rmsds[(id1, id2)] = np.nan

pcmd.delete('all')

rmsd_matrix = pd.DataFrame(
    [[align_rmsds.get((id1, id2), np.nan) for id2 in names] for id1 in names],
    index=names, columns=names
)
print('Alignment RMSD Matrix:')
print(rmsd_matrix.to_string(float_format='%.2f'))

### 정렬된 구조 3D 시각화

정렬된 모든 구조를 py3Dmol로 겹쳐서 표시합니다.
각 구조는 다른 색상으로 표시되며, 리간드도 함께 보여줍니다.
마우스로 회전/줌하여 결합 부위의 유사성과 차이를 확인하세요.


In [ ]:
#@title 3-2. 정렬된 구조 3D 시각화 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)
view.removeAllModels()

colors = ['#43A047', '#1E88E5', '#E53935', '#FF8F00', '#8E24AA',
          '#00ACC1', '#D81B60', '#6D4C41', '#546E7A', '#F4511E']

model_idx = 0
for i, pdb_id in enumerate(structures.keys()):
    prot_path = structures[pdb_id].get('prot_H_aligned', structures[pdb_id]['prot_H'])
    with open(prot_path) as f:
        pdb_str = f.read()
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'model': model_idx}, {'cartoon': {'color': colors[i % len(colors)], 'opacity': 0.7}})
    model_idx += 1

    # Ligand
    lig_path = structures[pdb_id].get('lig_aligned', structures[pdb_id]['lig_mol2'])
    if os.path.exists(lig_path):
        with open(lig_path) as f:
            lig_str = f.read()
        view.addModel(lig_str, 'mol2')
        view.setStyle({'model': model_idx}, {'stick': {'color': colors[i % len(colors)], 'radius': 0.2}})
        model_idx += 1

# Legend
legend = ' | '.join([f'<span style="color:{colors[i%len(colors)]}">{p.upper()}</span>' for i, p in enumerate(structures.keys())])
print(f'Colors: {", ".join([f"{p.upper()}={colors[i%len(colors)]}" for i, p in enumerate(structures.keys())])}')

view.zoomTo()
view.show()


### RMSD 히트맵

구조 간 정렬 RMSD를 히트맵으로 시각화합니다.
- 진한 색 = 큰 RMSD = 구조적 차이가 큼
- 연한 색 = 작은 RMSD = 구조적으로 유사


In [ ]:
#@title 3-3. RMSD 히트맵 {display-mode: "form"}
if len(rmsd_matrix) > 1:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(rmsd_matrix.astype(float), annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'RMSD (Å)'})
    ax.set_title(f'{TARGET_NAME} Structure Alignment RMSD')
    plt.tight_layout()
    plt.show()


## 4. Docking Box 설정 & 시각화

Docking box를 자동(ligand 기준) 또는 잔기 기준으로 설정합니다.


### Docking Box 설정

리간드가 탐색할 3차원 공간(Grid Box)을 정의합니다:
- **auto**: co-crystal 리간드 위치를 기준으로 자동 계산
- **residue**: 지정한 잔기 번호들의 중심으로 box 생성
- **manual**: 좌표를 직접 입력

**PADDING**: 리간드 bounding box 바깥으로 확장하는 여유 공간 (기본 7Å)


In [ ]:
#@title 4-1. Box 설정 방식 {display-mode: "form"}
BOX_METHOD = "auto"  #@param ["auto", "residue", "union", "manual"]
RESIDUE_LIST = "790, 858, 855"  #@param {type:"string"}
MANUAL_CENTER = "12.5, -3.0, 27.0"  #@param {type:"string"}
MANUAL_SIZE = "25.0, 25.0, 25.0"  #@param {type:"string"}
PADDING = 7.0  #@param {type:"number"}

def get_box_from_coords(coords, padding=7.0):
    minC = coords.min(axis=0)
    maxC = coords.max(axis=0)
    center = {'x': float((maxC[0]+minC[0])/2), 'y': float((maxC[1]+minC[1])/2), 'z': float((maxC[2]+minC[2])/2)}
    size = {'x': float(maxC[0]-minC[0]+2*padding), 'y': float(maxC[1]-minC[1]+2*padding), 'z': float(maxC[2]-minC[2]+2*padding)}
    return center, size

if BOX_METHOD == "auto":
    # 기준 구조의 co-crystal ligand 위치
    ref_lig = mda.Universe(structures[ref_name]['lig_mol2'])
    center, size = get_box_from_coords(ref_lig.atoms.positions, PADDING)
    print(f'Auto box from {ref_name.upper()} ligand')

elif BOX_METHOD == "union":
    # 모든 리간드 좌표 합쳐서 union box
    all_coords = []
    for pdb_id in structures.keys():
        try:
            u = mda.Universe(structures[pdb_id]['lig_mol2'])
            all_coords.append(u.atoms.positions)
        except: pass
    all_coords = np.vstack(all_coords)
    center, size = get_box_from_coords(all_coords, PADDING)
    print(f'Union box from {len(structures)} ligands')

elif BOX_METHOD == "residue":
    # 지정 잔기 중심
    res_nums = [int(r.strip()) for r in RESIDUE_LIST.split(',')]
    ref_prot = mda.Universe(structures[ref_name]['prot_H'])
    sel_str = ' or '.join([f'resid {r}' for r in res_nums])
    sel = ref_prot.select_atoms(sel_str)
    center, size = get_box_from_coords(sel.positions, PADDING)
    print(f'Residue box from residues {res_nums}')

elif BOX_METHOD == "manual":
    cx, cy, cz = [float(v.strip()) for v in MANUAL_CENTER.split(',')]
    sx, sy, sz = [float(v.strip()) for v in MANUAL_SIZE.split(',')]
    center = {'x': cx, 'y': cy, 'z': cz}
    size = {'x': sx, 'y': sy, 'z': sz}
    print('Manual box')

print(f'Center: ({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f})')
print(f'Size:   ({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')


### Box 3D 시각화

단백질 + 리간드 + Docking Box를 3D로 겹쳐서 보여줍니다.
- **파란 반투명 박스**: 도킹 탐색 공간
- 박스가 리간드/결합부위를 완전히 감싸는지 확인하세요
- 너무 크면 계산 시간 증가, 너무 작으면 포즈를 놓칠 수 있습니다


In [ ]:
#@title 4-2. Box 3D 시각화 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

# 기준 구조
with open(structures[ref_name]['prot_H']) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.6}})

# 기준 리간드
with open(structures[ref_name]['lig_mol2']) as f:
    view.addModel(f.read(), 'mol2')
view.setStyle({'model': 1}, {'stick': {'color': 'green', 'radius': 0.2}})

# Docking box (반투명 파란색)
view.addBox({
    'center': {'x': center['x'], 'y': center['y'], 'z': center['z']},
    'dimensions': {'w': size['x'], 'h': size['y'], 'd': size['z']},
    'color': 'blue', 'opacity': 0.15
})

# Box 외곽선
for dx in [-1, 1]:
    for dy in [-1, 1]:
        view.addCylinder({
            'start': {'x': center['x']+dx*size['x']/2, 'y': center['y']+dy*size['y']/2, 'z': center['z']-size['z']/2},
            'end': {'x': center['x']+dx*size['x']/2, 'y': center['y']+dy*size['y']/2, 'z': center['z']+size['z']/2},
            'radius': 0.1, 'color': 'blue', 'fromCap': False, 'toCap': False
        })

print(f'Blue box: center=({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f}), size=({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')
print('Box가 리간드를 완전히 감싸는지 확인하세요.')
print('너무 크거나 작으면 위 셀에서 PADDING 또는 방식을 조절하세요.')

view.zoomTo()
view.show()


## 5. 전체 구조 Re-docking

각 PDB의 co-crystal ligand를 동일한 docking box로 re-docking합니다.


### 전체 구조 Re-docking

각 PDB의 co-crystal 리간드를 SMILES에서 새로 3D 좌표를 생성한 뒤,
동일한 docking box로 re-docking합니다.

**Re-docking의 목적**: SMILES → 3D → 도킹한 포즈가 원래 crystal pose와 
얼마나 가까운지(RMSD) 검증하여 도킹 프로토콜의 신뢰성을 확인합니다.

**smina**: AutoDock Vina의 fork로, 추가 scoring function을 지원하며 CLI 기반으로 안정적입니다.


In [ ]:
#@title 5-1. Re-docking {display-mode: "form"}
EXHAUSTIVENESS = 8   #@param {type:"integer"}
N_POSES = 10         #@param {type:"integer"}
N_CPUS = 0           #@param {type:"integer"}

smina = os.path.join(BIN_DIR, 'smina')
t0 = time.time()

for i, pdb_id in enumerate(structures.keys()):
    s = structures[pdb_id]
    print(f'[{i+1}/{len(structures)}] {pdb_id.upper()}...', flush=True)

    # 1. Receptor PDBQT (openbabel + strip ROOT/BRANCH)
    rec_qt = os.path.join(s['dir'], f'{pdb_id}_rec.pdbqt')
    obmol = list(pybel.readfile(format='pdb', filename=s.get('prot_H_aligned', s['prot_H'])))[0]
    out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
    out.write(obmol); out.close()
    skip_tags = ('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')
    skip_kw = ('torsion','active','between atoms','status')
    with open(rec_qt+'.tmp') as f: raw = f.readlines()
    with open(rec_qt, 'w') as f:
        for l in raw:
            if l.startswith(skip_tags): continue
            if l.startswith('REMARK') and any(k in l.lower() for k in skip_kw): continue
            f.write(l)
    os.remove(rec_qt+'.tmp')
    print(f'  Receptor PDBQT OK', flush=True)

    # 2. Ligand SDF (SMILES → RDKit 3D → SDF)
    lig_sdf = os.path.join(s['dir'], f'{pdb_id}_lig_redock.sdf')
    try:
        lmol = list(pybel.readfile(format='mol2', filename=s['lig_mol2']))[0]
        smi = lmol.write('smi').split()[0]
        rdmol = Chem.MolFromSmiles(smi)
        if rdmol:
            rdmol = Chem.AddHs(rdmol)
            AllChem.EmbedMolecule(rdmol, randomSeed=42)
            AllChem.MMFFOptimizeMolecule(rdmol)
            writer = Chem.SDWriter(lig_sdf)
            writer.write(rdmol)
            writer.close()
            print(f'  Ligand SDF OK ({smi[:30]}...)', flush=True)
        else:
            print(f'  Ligand SKIP (invalid SMILES)')
            continue
    except Exception as e:
        print(f'  Ligand SKIP ({e})')
        continue

    # 3. smina docking (SDF input directly)
    sdf_out = os.path.join(s['dir'], f'{pdb_id}_docked.sdf')
    print(f'  Docking (exhaustiveness={EXHAUSTIVENESS})...', flush=True)
    subprocess.run([
        smina, '-r', rec_qt, '-l', lig_sdf, '-o', sdf_out,
        '--center_x', str(center['x']), '--center_y', str(center['y']), '--center_z', str(center['z']),
        '--size_x', str(size['x']), '--size_y', str(size['y']), '--size_z', str(size['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
        '--cpu', str(N_CPUS),
    ], stdout=None, stderr=None)

    s['rec_qt'] = rec_qt
    s['lig_sdf'] = lig_sdf
    s['sdf_out'] = sdf_out

    # 4. Parse ALL poses
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        scores = []
        for mol in suppl:
            if mol is None: continue
            props = mol.GetPropsAsDict()
            if 'minimizedAffinity' in props:
                try: scores.append(float(props['minimizedAffinity']))
                except: pass
        s['all_scores'] = scores
        s['best_score'] = min(scores) if scores else None
        s['n_poses'] = len(scores)
        if scores:
            print(f'  → {len(scores)} poses: best={min(scores):.2f}, worst={max(scores):.2f}, mean={np.mean(scores):.2f}')
        else:
            print(f'  → No valid poses')
    else:
        print(f'  → Docking FAILED')

elapsed = time.time() - t0
print(f'\n=== {len([s for s in structures.values() if "best_score" in s])}/{len(structures)} docked in {elapsed:.0f}s ===')

## 6. 분석


### Re-docking RMSD 검증

도킹된 최적 포즈와 원래 crystal 포즈 간의 RMSD를 계산합니다.
- **RMSD < 2.0Å**: 성공적 re-docking (도킹 프로토콜 신뢰 가능)
- **RMSD 2.0~3.0Å**: 주의 (부분적 일치)
- **RMSD > 3.0Å**: 실패 (이 구조/프로토콜은 부적합)


In [ ]:
#@title 6-1. Re-docking RMSD 검증 (전체 포즈) {display-mode: "form"}
for pdb_id in structures.keys():
    s = structures[pdb_id]
    sdf_out = s.get('sdf_out', '')
    if not os.path.exists(sdf_out): continue

    # Reference: crystal ligand
    ref_mol = None
    try:
        ref_mol = Chem.MolFromMol2File(s.get('lig_aligned', s['lig_mol2']), removeHs=False, sanitize=False)
        if ref_mol:
            Chem.SanitizeMol(ref_mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
    except: pass

    if ref_mol is None:
        s['rmsds'] = []
        s['best_rmsd'] = np.nan
        continue

    try:
        ref_noH = Chem.RemoveHs(ref_mol)
    except:
        s['rmsds'] = []
        s['best_rmsd'] = np.nan
        continue

    # Calculate RMSD for ALL poses
    rmsds = []
    suppl = Chem.SDMolSupplier(sdf_out, removeHs=False, sanitize=False)
    for pose_idx, mol in enumerate(suppl):
        if mol is None: continue
        try:
            Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
            docked_noH = Chem.RemoveHs(mol)
            if ref_noH.GetNumAtoms() == docked_noH.GetNumAtoms():
                rmsd = rdMolAlign.GetBestRMS(ref_noH, docked_noH)
                rmsds.append(round(rmsd, 2))
            else:
                rmsds.append(np.nan)
        except:
            rmsds.append(np.nan)

    s['rmsds'] = rmsds
    s['best_rmsd'] = min([r for r in rmsds if not np.isnan(r)]) if rmsds and any(not np.isnan(r) for r in rmsds) else np.nan
    s['best_rmsd_pose'] = rmsds.index(s['best_rmsd']) if not np.isnan(s['best_rmsd']) else -1

    valid = [r for r in rmsds if not np.isnan(r)]
    if valid:
        print(f"{pdb_id.upper()}: {len(valid)} poses — best RMSD={min(valid):.2f}A (pose {s['best_rmsd_pose']+1}), mean={np.mean(valid):.2f}A, <2A: {sum(1 for r in valid if r < 2.0)}/{len(valid)}")
    else:
        print(f"{pdb_id.upper()}: RMSD calculation failed (atom mismatch)")

### ProLIF 상호작용 분석

ProLIF (Protein-Ligand Interaction Fingerprints)로 
도킹 포즈와 단백질 잔기 간의 상호작용을 자동으로 분석합니다:
- **수소결합 (HBDonor/HBAcceptor)**: 약물 결합의 핵심
- **소수성 (Hydrophobic)**: 포켓 내 소수성 접촉
- **π-π 스태킹 (PiStacking)**: 방향족 고리 간 상호작용


In [ ]:
#@title 6-2. ProLIF 상호작용 분석 (전체 포즈) {display-mode: "form"}
for pdb_id in structures.keys():
    s = structures[pdb_id]
    sdf_out = s.get('sdf_out', '')
    if not os.path.exists(sdf_out) or os.path.getsize(sdf_out) < 100: continue

    try:
        prot_file = s.get('prot_H_aligned', s['prot_H'])
        prot_mol = Chem.MolFromPDBFile(prot_file, removeHs=False, sanitize=False)
        if prot_mol:
            Chem.SanitizeMol(prot_mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
            prot_plf = plf.Molecule(prot_mol)
        else:
            continue

        suppl = Chem.SDMolSupplier(sdf_out, sanitize=False, removeHs=False)
        ligs = []
        for mol in suppl:
            if mol is None: continue
            try:
                Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
                ligs.append(plf.Molecule(mol))
            except: pass

        if ligs:
            fp = plf.Fingerprint()
            fp.run_from_iterable(ligs, prot_plf, n_jobs=1)
            df_ifp = fp.to_dataframe()

            # Count interaction types across ALL poses
            interactions = {}
            conserved = {}  # interactions present in >50% of poses
            for col in df_ifp.columns:
                itype = str(col[2]) if isinstance(col, tuple) and len(col) > 2 else str(col)
                residue = str(col[1]) if isinstance(col, tuple) and len(col) > 1 else ''
                freq = df_ifp[col].sum() / len(df_ifp)
                if df_ifp[col].any():
                    interactions[itype] = interactions.get(itype, 0) + 1
                if freq >= 0.5:
                    conserved[f'{residue}_{itype}'] = round(freq, 2)

            s['interactions'] = interactions
            s['n_interactions'] = sum(interactions.values())
            s['conserved'] = conserved
            s['n_conserved'] = len(conserved)
            s['n_poses_analyzed'] = len(ligs)
            print(f'{pdb_id.upper()}: {len(ligs)} poses analyzed — {s["n_interactions"]} total interactions, {s["n_conserved"]} conserved (>50%)')
            if conserved:
                top5 = sorted(conserved.items(), key=lambda x: -x[1])[:5]
                print(f'  Top conserved: {", ".join(f"{k}({v:.0%})" for k,v in top5)}')
    except Exception as e:
        print(f'{pdb_id.upper()}: ProLIF failed - {e}')
        s['interactions'] = {}
        s['n_interactions'] = 0
        s['conserved'] = {}
        s['n_conserved'] = 0

### 종합 비교 & 판정

모든 분석 결과를 종합하여 각 PDB 구조를 평가합니다:
- **✓ 추천**: Score < -7.0, RMSD < 2.0, 상호작용 3개 이상
- **△ 가능**: Score < -6.0, RMSD < 3.0
- **✗ 부적합**: 기준 미달


In [ ]:
#@title 6-3. 종합 비교 테이블 {display-mode: "form"}
comparison = []
for pdb_id in structures.keys():
    s = structures[pdb_id]
    info = structs_df[structs_df['PDB'] == pdb_id.upper()]
    res = info['Resolution'].values[0] if len(info) > 0 else np.nan
    lig = info['Ligand'].values[0] if len(info) > 0 else '?'

    score = s.get('best_score', np.nan)
    n_poses = s.get('n_poses', 0)
    rmsd = s.get('best_rmsd', np.nan)
    rmsd_pose = s.get('best_rmsd_pose', -1) + 1
    n_int = s.get('n_interactions', 0)
    n_cons = s.get('n_conserved', 0)

    # Verdict
    if rmsd < 2.0 and score < -7.0 and n_cons > 2:
        verdict = '✓ 추천'
    elif rmsd < 3.0 and score < -6.0:
        verdict = '△ 가능'
    elif np.isnan(rmsd):
        verdict = '? RMSD 불가'
    else:
        verdict = '✗ 부적합'

    comparison.append({
        'PDB': pdb_id.upper(), 'Resolution': res, 'Ligand': lig,
        'Best Score': score, 'Poses': n_poses,
        'Best RMSD': rmsd, 'RMSD Pose#': rmsd_pose,
        'Interactions': n_int, 'Conserved': n_cons,
        'Verdict': verdict
    })

comp_df = pd.DataFrame(comparison)
print('=== 종합 비교 ===')
comp_df

### 비교 차트

Score, RMSD, 상호작용 수를 3개 차트로 나란히 비교합니다.
한눈에 어떤 구조가 가장 적합한지 파악할 수 있습니다.


In [ ]:
#@title 6-4. 비교 차트 {display-mode: "form"}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Score comparison
if 'Score' in comp_df.columns:
    colors = ['#43A047' if v == '✓ 추천' else '#FFB300' if v == '△ 가능' else '#E53935' for v in comp_df['Verdict']]
    axes[0].barh(comp_df['PDB'], comp_df['Score'], color=colors, edgecolor='black', linewidth=0.5)
    axes[0].set_xlabel('Docking Score (kcal/mol)')
    axes[0].set_title('Docking Score')
    axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)

# RMSD
if 'RMSD' in comp_df.columns:
    colors = ['#43A047' if r < 2.0 else '#FFB300' if r < 3.0 else '#E53935' for r in comp_df['RMSD'].fillna(9)]
    axes[1].barh(comp_df['PDB'], comp_df['RMSD'].fillna(0), color=colors, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('RMSD (Å)')
    axes[1].set_title('Re-docking RMSD')
    axes[1].axvline(x=2.0, color='red', linestyle='--', alpha=0.5, label='2.0 Å')
    axes[1].legend()

# Interactions
if 'Interactions' in comp_df.columns:
    axes[2].barh(comp_df['PDB'], comp_df['Interactions'], color='#1E88E5', edgecolor='black', linewidth=0.5)
    axes[2].set_xlabel('# Interactions')
    axes[2].set_title('ProLIF Interactions')

plt.tight_layout()
plt.show()


### 도킹 포즈 3D 비교

각 PDB에서 도킹된 최적 포즈를 겹쳐서 3D로 비교합니다.
색상별로 어떤 구조의 포즈인지 구분할 수 있습니다.


In [ ]:
#@title 6-5. Best 포즈 3D 비교 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

# Reference protein
with open(structures[ref_name]['prot_H']) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.5}})

# Docked poses from each PDB
model_idx = 1
for i, pdb_id in enumerate(structures.keys()):
    if pdb_id not in structures: continue
    sdf_out = structures[pdb_id].get('sdf_out', '')
    if not os.path.exists(sdf_out) or os.path.getsize(sdf_out) < 100: continue
    try:
        suppl = Chem.SDMolSupplier(sdf_out, removeHs=False, sanitize=False)
        if suppl and suppl[0]:
            block = Chem.MolToMolBlock(suppl[0])
            view.addModel(block, 'mol')
            view.setStyle({'model': model_idx}, {'stick': {'color': colors[i % len(colors)], 'radius': 0.2}})
            model_idx += 1
    except: pass

legend = ', '.join([f'{p.upper()}={colors[i%len(colors)]}' for i, p in enumerate(structures.keys())])
print(f'Docked poses: {legend}')
view.zoomTo()
view.show()

## 7. 결론 & 내보내기


### 최적 구조 판정

종합 평가에서 "추천" 판정을 받은 구조 중 가장 좋은 것을 선택합니다.
이 구조를 후속 노트북(02_Ligand_Screening, 03_SAR_Matrix)에서 사용합니다.


In [ ]:
#@title 7-1. 최적 구조 판정 {display-mode: "form"}
recommended = comp_df[comp_df['Verdict'] == '✓ 추천']
if len(recommended) > 0:
    best = recommended.sort_values('Score').iloc[0]
    print(f'=== 추천 구조: {best["PDB"]} ===')
    print(f'  Resolution: {best["Resolution"]:.1f} Å')
    print(f'  Docking Score: {best["Score"]:.2f} kcal/mol')
    print(f'  Re-docking RMSD: {best["RMSD"]:.2f} Å')
    print(f'  Interactions: {best["Interactions"]}')
    print(f'\n→ 이 PDB를 노트북 2 (리간드 스크리닝), 노트북 3 (SAR 매트릭스)에서 사용하세요.')
    BEST_PDB = best['PDB'].lower()
else:
    print('추천할 만한 구조가 없습니다.')
    print('Resolution이 2.5Å 이하이고, co-crystal ligand가 있는 다른 PDB를 시도하세요.')
    BEST_PDB = ref_name


### 결과 내보내기 & 다운로드

분석 결과를 파일로 저장합니다:
- **CSV**: 비교 테이블 (DataWarrior/Excel에서 열기)
- **PML**: PyMOL 스크립트 (정렬된 구조를 3D로 보기)
- **ZIP**: 전체 결과 압축 파일


In [ ]:
#@title 7-2. 결과 다운로드 {display-mode: "form"}
import shutil

# Save comparison table
comp_csv = os.path.join(WORK_DIR, 'structure_comparison.csv')
comp_df.to_csv(comp_csv, index=False)
print(f'Comparison CSV: {comp_csv}')

# PyMOL script for aligned structures
pml_path = os.path.join(WORK_DIR, 'aligned_structures.pml')
with open(pml_path, 'w') as f:
    f.write('reinitialize\nbg_color white\n\n')
    for i, pdb_id in enumerate(structures.keys()):
        if pdb_id not in structures: continue
        s = structures[pdb_id]
        prot = s.get('prot_H_aligned', s['prot_H'])
        sdf = s.get('sdf_out', '')
        c = ['palegreen','lightblue','salmon','lightorange','lightpink'][i%5]
        f.write(f'load {prot}, {pdb_id}\n')
        f.write(f'color {c}, {pdb_id}\n')
        f.write(f'show cartoon, {pdb_id}\n')
        if os.path.exists(sdf):
            f.write(f'load {sdf}, {pdb_id}_dock\n')
            f.write(f'set all_states, 0\nframe 1\n')
            f.write(f'show sticks, {pdb_id}_dock\n\n')
print(f'PyMOL script: {pml_path}')

# Zip
zip_path = os.path.join(os.path.dirname(WORK_DIR), f'{TARGET_NAME}_structure_search')
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'\nArchive: {zip_path}.zip')

try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_DIR}')
